# 1. Konfiguracja środowiska oraz datasetu

## 1.1. Instalacja zależności

In [ ]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#%pip install -r requirements.txt

## 1.2.Konfiguracja importów

In [ ]:
import random
import torch
from torch.utils.data import DataLoader
from pathlib import Path
from IPython.display import Audio
from src.process_guitarset import process_dataset
from src.config import *
from src.dataset import GuitarSetDataset, collate_fn
from src.model import MultiTaskTranscriptionModel
from src.train import train_one_epoch, evaluate
import torch.optim as optim
from src.visualization import (
    plot_audio_waveform,
    plot_cqt,
    plot_annotation_map,
)
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

## 1.3. Konfiguracja GPU - automatyczne wykrywanie

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

## 1.4. Wczytanie guitarsetu

In [ ]:
config = {
    "data_dir": "src/guitarset_data",
    "output_dir": "src/processed_data",
    "seed": 42,
    "apply_augmentation": False,
    "max_tracks": None,
    "overwrite": False
}

process_dataset(**config)

## 1.5. Wczytanie guitarseta z plików batch

In [ ]:
# Wczytanie danych z plików
train_dir = Path(config["output_dir"]) / "train"
val_dir = Path(config["output_dir"]) / "val"
test_dir = Path(config["output_dir"]) / "test"

train_dataset = GuitarSetDataset(train_dir)
val_dataset = GuitarSetDataset(val_dir)
test_dataset = GuitarSetDataset(test_dir)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

# sprawdzanie notes, onsets, contours w train_loader
batch = next(iter(train_loader))

notes = batch['notes']    # [B, T, F_notes]
onsets = batch['onsets']  # [B, T, F_notes]
contours = batch['contours']  # [B, T, F_contours]
mask_expanded = batch['mask'].unsqueeze(-1)  # [B, T, 1]

notes_masked = (notes > 0) & (mask_expanded > 0)
onsets_masked = (onsets > 0) & (mask_expanded > 0)
contours_masked = (contours > 0) & (mask_expanded > 0)

n_notes_pos = notes_masked.sum().item()
n_onsets_pos = onsets_masked.sum().item()
n_contours_pos = contours_masked.sum().item()

# Liczba niepaddingowanych ramek (czasowych pozycji)
valid_timesteps = batch['mask'].sum(dim=1)  # [B]
total_valid_positions = valid_timesteps.sum().item()

# Mnożymy przez liczbę cech
total_notes = total_valid_positions * notes.shape[2]
total_contours = total_valid_positions * contours.shape[2]

print(f"Notes positives: {n_notes_pos} / {total_notes} ({n_notes_pos / total_notes:.6f})")
print(f"Onsets positives: {n_onsets_pos} / {total_notes} ({n_onsets_pos / total_notes:.6f})")
print(f"Contours positives: {n_contours_pos} / {total_contours} ({n_contours_pos / total_contours:.6f})")


## 1.6. Wizualizacja przykładowego pliku

In [ ]:
# Wybierz przykładową próbkę z batcha
sample_idx = random.randint(0, batch["audio"].shape[0] - 1)

# Parametry audio
sr = AUDIO_SAMPLE_RATE
hop_length = FFT_HOP

# Maska – tylko ważne ramki (bez paddingu)
mask = batch["mask"][sample_idx].bool()
T_valid = mask.sum().item()

# Przycięte dane
audio = batch["audio"][sample_idx].numpy()
audio_trimmed = audio[:T_valid * hop_length]

cqt = batch["features"][sample_idx][mask].numpy()  # [T_valid, F]
onset_map = batch["onsets"][sample_idx][mask]      # [T_valid, F_notes]
notes_map = batch["notes"][sample_idx][mask].T.numpy()       # [F_notes, T_valid]
contours_map = batch["contours"][sample_idx][mask].T.numpy() # [F_contours, T_valid]

# Wyznaczenie onsetów (indeksy czasowe z wartościami > 0)
onset_presence = (onset_map.sum(dim=1) > 0).nonzero(as_tuple=True)[0]

# Wizualizacje
plot_audio_waveform(audio_trimmed, sr=sr, onset_frames=onset_presence.numpy(), hop_length=hop_length)
plot_cqt(cqt, sr=sr, hop_length=hop_length, freq_bins=FREQ_BINS_NOTES)
plot_annotation_map(notes_map, title="Adnotacje nut (sparse piano roll)",
                    ylabel="Bin częstotliwości (nuty)", cmap='hot', sr=sr, hop_length=hop_length)

plot_annotation_map(contours_map, title="Adnotacje konturów (multif0)",
                    ylabel="Bin częstotliwości (kontury)", cmap='Blues', sr=sr, hop_length=hop_length)

# Odtwarzanie przyciętego audio
Audio(audio_trimmed, rate=sr)


# 2. Model i uczenie

## 2.1. Wczytanie implementacji modelu

In [ ]:
# Przykładowy forward pass
model = MultiTaskTranscriptionModel(
    freq_bins_out_notes=N_FREQ_BINS_NOTES,
    freq_bins_out_contours=N_FREQ_BINS_CONTOURS,
    hidden_channels=64
).to(device)

## 2.2. Konfiguracja treningu

In [ ]:
from src.train import run_training

train_config = {
    "lr": 1e-4,
    "epochs": 50,
    "gamma": 2.0,
    "alpha_notes": 0.75,
    "alpha_onsets": 0.9,
    "alpha_contours": 0.5,
    "patience": 5,
    "output_dir": "models"
}

# Uruchomienie treningu
trained_model, history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    **train_config
)